In [1]:
# %%
### ValDXer testing
import os

# os.environ["HDXER_PATH"] = "/homes/hussain/HDXer"
os.environ["HDXER_PATH"] = "/home/alexi/Documents/HDXer"

import sys

sys.path.append("/home/alexi/Documents/ValDX/")


import MDAnalysis as mda

from ValDX.ValidationDX import ValDXer
from ValDX.VDX_Settings import Settings

# settings.stride = 1000
# # settings.HDXer_stride = 10000

# settings.RW_do_reweighting = False
# settings.RW_do_params = True



/home/alexi/Documents/ValDX/figure_scripts/jaxent_cross_validation/BPTI


/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/Bio/Application/__init__.py:40: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [2]:

# %%

# %%
# import subprocess
# from ValDX.helpful_funcs import conda_to_env_dict

# # Assuming settings.HDXer_env contains the name of your Conda environment
# env_path = conda_to_env_dict(settings.HDXer_env)

# command = "echo $HDXER_PATH"
# print("command:", command)

# # Run the command in the subprocess
# output = subprocess.run(command, shell=True, env=env_path, capture_output=True, text=True)

# # Capture and print the standard output (stdout)
# hdxer_path = output.stdout.strip()  # .strip() removes any trailing newline
# print("HDXER_PATH:", hdxer_path)


def MD_traj_to_interval_paths(top_path, traj_path, n_intervals=10, n_reps=None, out_path=None):
    top_name = os.path.basename(top_path).replace(".pdb", "")

    if n_reps is None:
        # extract the number of replicates from the top_name
        split = top_name.split("_")
        # find split that starts with "r"
        for s in split:
            if s.startswith("r"):
                if s[1:].isdigit():
                    n_reps = int(s[1:])
                    break

    if out_path is None:
        out_path = os.path.join(os.path.dirname(traj_path), "time_intervals")

    os.makedirs(out_path, exist_ok=True)

    # load the trajectory

    u = mda.Universe(top_path, traj_path)

    length = len(u.trajectory)

    # check that the number of frames is divisible by n_reps

    assert length % n_reps == 0, f"Number of frames {length} is not divisible by n_reps {n_reps}"

    traj_length = length / n_reps

    # round down interval length to the nearest integer
    interval_length = int(traj_length / n_intervals)

    # universe represents a concatenated trajectory for each replicate
    # when slicing into intervals the frames must be selected from each replicate
    # in a way that the intervals are continuous in time
    # create a new trajectory for each interval

    interval_indexes = {i: [] for i in range(n_intervals)}

    for i in range(n_intervals):
        start = i * interval_length
        end = (i + 1) * interval_length

        for j in range(n_reps):
            interval_indexes[i] += list(
                range(int(j * traj_length + start), int(j * traj_length + end))
            )

    # create a new trajectory for each interval
    print(interval_indexes[0])

    interval_names = [
        f"{top_name}_nI{n_intervals}_interval{i}_len{n_intervals * interval_length}" + ".xtc"
        for i in range(n_intervals)
    ]

    new_traj_paths = []

    for i, indexes in interval_indexes.items():
        new_traj_path = os.path.join(out_path, interval_names[i])
        new_traj_paths.append([new_traj_path])

    top_paths = [top_path] * n_intervals

    return new_traj_paths, top_paths



In [3]:

def pre_process_main_BPTI():
    # BPTI data
    expt_name = "Experimental"
    test_name = "BPTI_af_rank1"
    test_names = [
        "BPTI_TFES",
        "BPTI_af_dirty",
        "BPTI_af_clean",
        "BPTI_shaw_400",
        "BPTI_MD_Bad",
        "BPTI_MD_Good",
        "BPTI_MD_Good+Bad",
    ]

    BPTI_dir = "/home/alexi/Documents/Library/CloudStorage/OneDrive-Nexus365/Rotation_Projects/Rotation_3/Project/ValDX/raw_data/HDXer_tutorial/BPTI"
    BPTI_dir = "/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI"
    # BPTI_dir = "/data/localhost/not-backed-up/hussain/ValDX/raw_data/HDXer_tutorial/BPTI"
    # BPTI_dir = "/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI"
    expt_dir = os.path.join(BPTI_dir, "BPTI_expt_data")

    os.listdir(expt_dir)

    segs_name = "BPTI_residue_segs_trimmed.txt"
    segs_path = os.path.join(expt_dir, segs_name)

    hdx_name = "BPTI_expt_dfracs_clean_trimmed.dat"
    hdx_path = os.path.join(expt_dir, hdx_name)
    print(hdx_path)

    rates_name = "BPTI_Intrinsic_rates.dat"
    rates_path = os.path.join(expt_dir, rates_name)
    sim_name = "BPTI_MD"

    sim_dir = os.path.join(BPTI_dir, "BPTI_simulations")

    os.listdir(sim_dir)

    md_reps = 1
    rep_dirs = ["Run_" + str(i + 1) for i in range(md_reps)]

    top_name = "bpti_5pti_eq6_protonly.gro"

    top_path = os.path.join(sim_dir, rep_dirs[0], top_name)

    traj_name = "bpti_5pti_reimg_protonly.xtc"

    traj_paths = [os.path.join(sim_dir, rep_dir, traj_name) for rep_dir in rep_dirs]

    print(top_path)
    print(traj_paths)

    dirty_top_path = "/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/P00974_60_1_af_sample_127_10000_protonated.pdb"
    # top_path =  "/data/localhost/not-backed-up/hussain/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/P00974_60_1_af_sample_127_10001_protonated.pdb"
    dirty_traj_paths = [
        "/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/P00974_60_1_af_sample_127_10000_protonated.xtc"
    ]
    # traj_paths = ["/data/localhost/not-backed-up/hussain/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/P00974_60_1_af_sample_127_10001_protonated.xtc"]

    clean_top_path = "/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/P00974_60_1_af_sample_127_10001_protonated.pdb"
    clean_traj_paths = [
        "/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/P00974_60_1_af_sample_127_10000_protonated_all_filtered.xtc"
    ]

    shaw_top_path = (
        "/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/SHAW/bpti.pdb"
    )
    shaw_traj_paths = [
        "/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/SHAW/reduced_BPTI_SHAW_stride_400.xtc"
    ]

    badMD_top_path = (
        "/home/alexi/Documents/ValDX/raw_data/good_bad_MD/BPTI/BadMD_BPTI_r5_15010_concatenated.pdb"
    )
    badMD_traj_path = [badMD_top_path.replace(".pdb", ".xtc")]

    goodMD_top_path = "/home/alexi/Documents/ValDX/raw_data/good_bad_MD/BPTI/GoodMD_BPTI_r10_10010_concatenated.pdb"
    goodMD_traj_path = [goodMD_top_path.replace(".pdb", ".xtc")]

    topology_path = (
        "/home/alexi/Documents/JAX-ENT/tests/inst/clean/BPTI/BPTI_overall_combined_stripped.pdb"
    )
    trajectory_path = "/home/alexi/Documents/JAX-ENT/tests/inst/clean/BPTI/BPTI_sampled_500.xtc"

    top_paths = [
        topology_path,
        # dirty_top_path,
        # clean_top_path,
        # shaw_top_path,
        # badMD_top_path,
        # goodMD_top_path,
        # goodMD_top_path,
    ]
    traj_paths = [
        trajectory_path,
        # dirty_traj_paths[0],
        # clean_traj_paths[0],
        # shaw_traj_paths[0],
        # badMD_traj_path[0],
        # goodMD_traj_path[0],
        # [goodMD_traj_path[0], badMD_traj_path[0]],
    ]

    min_interval_size = 500
    confidence_intervals = [
        (0.0, 0.1),
        (0.1, 0.2),
        (0.2, 0.3),
        (0.3, 0.4),
        (0.4, 0.5),
        (0.5, 0.6),
        (0.6, 0.7),
        (0.7, 0.8),
        (0.8, 0.9),
        (0.9, 1.0),
        ("top", min_interval_size),
        ("bottom", min_interval_size),
    ]
    str_confidence_intervals = [f"{i}_{j}" for (i, j) in confidence_intervals]
    conf_interval_names = [f"BPTI_af_conf{i}" for i in str_confidence_intervals]
    conf_interval_traj_names = [
        dirty_traj_path.replace(".xtc", f"_{name}.xtc")
        for name in str_confidence_intervals
        for dirty_traj_path in dirty_traj_paths
    ]
    conf_dir = "af_confidence_intervals"
    conf_interval_paths = [
        os.path.join(os.path.dirname(dirty_top_path), conf_dir, os.path.basename(name))
        for name in conf_interval_traj_names
    ]

    BadMD_interval_traj_paths, BadMD_interval_top_paths = MD_traj_to_interval_paths(
        badMD_top_path, badMD_traj_path[0]
    )
    BadMD_interval_test_names = [f"BPTI_MD_Bad-Int{i}" for i in range(10)]
    GoodMD_interval_traj_paths, GoodMD_interval_top_paths = MD_traj_to_interval_paths(
        goodMD_top_path, goodMD_traj_path[0]
    )
    GoodMD_interval_test_names = [f"BPTI_MD_Good-Int{i}" for i in range(10)]

    # test_names = test_names + conf_interval_names
    # top_paths = top_paths + [dirty_top_path]*len(conf_interval_names)
    # traj_paths = traj_paths + conf_interval_paths

    # top_paths = BadMD_interval_top_paths + GoodMD_interval_top_paths
    # traj_paths = BadMD_interval_traj_paths + GoodMD_interval_traj_paths
    # test_names = BadMD_interval_test_names + GoodMD_interval_test_names

    return hdx_path, segs_path, rates_path, top_paths, traj_paths, sim_name, expt_name, test_names


# %%
hdx_path, segs_path, rates_path, top_paths, traj_paths, sim_name, expt_name, test_names = (
    pre_process_main_BPTI()
)

# %%


/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_expt_data/BPTI_expt_dfracs_clean_trimmed.dat
/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/Run_1/bpti_5pti_eq6_protonly.gro
['/home/alexi/Documents/ValDX/raw_data/HDXer_tutorial/BPTI/BPTI_simulations/Run_1/bpti_5pti_reimg_protonly.xtc']
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 15

/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/MDAnalysis/lib/mdamath.py:300: RuntimeWarning: invalid value encountered in scalar divide
  alpha = np.rad2deg(np.arccos(np.dot(y, z) / (ly * lz)))
/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/MDAnalysis/lib/mdamath.py:301: RuntimeWarning: invalid value encountered in scalar divide
  beta = np.rad2deg(np.arccos(np.dot(x, z) / (lx * lz)))
/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/MDAnalysis/lib/mdamath.py:302: RuntimeWarning: invalid value encountered in scalar divide
  gamma = np.rad2deg(np.arccos(np.dot(x, y) / (lx * ly)))


In [ ]:


# %%
times = [0.167, 1, 10]

for idx, (test_name, top_path, traj_paths) in enumerate(
    zip(test_names[:1], top_paths[:1], traj_paths[:1])
):
    # if idx < 5:
    #     continue

    settings = Settings(name=test_name)
    # settings.replicates = 2
    settings.gamma_range = (1, 8)
    settings.train_frac = 0.5
    settings.RW_exponent = [0]
    # settings.split_mode = 'R3'

    VDX = ValDXer(settings)

    # run RW across all splits

    combined_analysis_dump, names, save_paths = VDX.run_benchmark_ensemble(
        system=test_name,
        times=times,
        split_modes=["r","Sp"],
        expt_name=expt_name,
        n_reps=5,
        RW=True,
        hdx_path=hdx_path,
        segs_path=segs_path,
        traj_paths=[traj_paths],
        top_path=top_path,
    )


envs ['# conda environments:', '#', 'base                     /home/alexi/anaconda3', 'FES                      /home/alexi/anaconda3/envs/FES', 'HDXER_ENV             *  /home/alexi/anaconda3/envs/HDXER_ENV', 'PLUMED_310               /home/alexi/anaconda3/envs/PLUMED_310', 'aider                    /home/alexi/anaconda3/envs/aider', 'automatic_sentiment      /home/alexi/anaconda3/envs/automatic_sentiment', 'boltz                    /home/alexi/anaconda3/envs/boltz', 'boltz_msa                /home/alexi/anaconda3/envs/boltz_msa', 'hdx_topology             /home/alexi/anaconda3/envs/hdx_topology', 'mood_modelling           /home/alexi/anaconda3/envs/mood_modelling', 'openfold_env             /home/alexi/anaconda3/envs/openfold_env', 'plumed-masterclass-2022     /home/alexi/anaconda3/envs/plumed-masterclass-2022', 'pydeps                   /home/alexi/anaconda3/envs/pydeps', '                         /home/alexi/localcolabfold/colabfold-conda', '                         /home/alexi/loc

/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarni

Selecting [0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11] peptides from the dataframe
Resnumbers calculated for 11 segments
Residues calculated for 11 segments
Selecting [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 29, 31, 32, 33, 34, 35] peptides from the dataframe
Resnumbers calculated for 22 segments
Residues calculated for 22 segments
Found intersecting residues: []
Selecting peptides with residues []
Selecting [] peptides from the dataframe
Found intersecting peptides: []
Train peptides: [ 0  1  2  4  5  6  7  8  9 10 11]
Val peptides: [12 13 14 15 16 17 18 19 20 21 22 23 24 25 27 28 29 31 32 33 34 35]
Intersection %: 0.00
Train %: 0.33
Selecting [0, 1, 2, 4, 5, 6, 7, 8, 9, 10, 11] peptides from the dataframe
Selecting [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 27, 28, 29, 31, 32, 33, 34, 35] peptides from the dataframe
train_segs
   ResStr  ResEnd  peptide          calc_name
0       4       5        0  train_BPTI_TFES_3
1       5       6        1  train_BPTI

/home/alexi/anaconda3/envs/HDXER_ENV/lib/python3.8/site-packages/sklearn/cluster/_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


envs ['# conda environments:', '#', 'base                     /home/alexi/anaconda3', 'FES                      /home/alexi/anaconda3/envs/FES', 'HDXER_ENV             *  /home/alexi/anaconda3/envs/HDXER_ENV', 'PLUMED_310               /home/alexi/anaconda3/envs/PLUMED_310', 'aider                    /home/alexi/anaconda3/envs/aider', 'automatic_sentiment      /home/alexi/anaconda3/envs/automatic_sentiment', 'boltz                    /home/alexi/anaconda3/envs/boltz', 'boltz_msa                /home/alexi/anaconda3/envs/boltz_msa', 'hdx_topology             /home/alexi/anaconda3/envs/hdx_topology', 'mood_modelling           /home/alexi/anaconda3/envs/mood_modelling', 'openfold_env             /home/alexi/anaconda3/envs/openfold_env', 'plumed-masterclass-2022     /home/alexi/anaconda3/envs/plumed-masterclass-2022', 'pydeps                   /home/alexi/anaconda3/envs/pydeps', '                         /home/alexi/localcolabfold/colabfold-conda', '                         /home/alexi/loc

...

concat plot_df
plotting nan_df
nan_df


ic| nan_df:         0.167       1.0      10.0  peptide          calc_name  ResStr  ResEnd  \
            0    0.169520  0.671190  0.999990        0  train_BPTI_TFES_1     NaN     NaN   
            1    0.064750  0.330250  0.981840        1  train_BPTI_TFES_1     NaN     NaN   
            2    0.277470  0.857170  1.000000        2  train_BPTI_TFES_1     NaN     NaN   
            3    0.767620  0.999840  1.000000        3  train_BPTI_TFES_1     NaN     NaN   
            4    0.967440  1.000000  1.000000        4  train_BPTI_TFES_1     NaN     NaN   
            ..        ...       ...       ...      ...                ...     ...     ...   
            325  0.118348  0.529632  0.999470       19    val_BPTI_TFES_5    31.0    32.0   
            326  0.001566  0.009338  0.089557       20    val_BPTI_TFES_5    32.0    33.0   
            327  0.178849  0.692700  0.999992       21    val_BPTI_TFES_5    33.0    34.0   
            328  0.018274  0.104558  0.668583       23    val_BPTI_TFE

nan_df + expt_df
plotting MSE for nan_df


NaN
          347  0.99999  1.00000  1.00000       17  Experimental     NaN     NaN  NaN
          348  0.00028  0.00166  0.01649       18  Experimental     NaN     NaN  NaN
          349  0.61477  0.99669  1.00000       19  Experimental     NaN     NaN  NaN
          350  0.00033  0.00198  0.01966       20  Experimental     NaN     NaN  NaN
          351  0.68572  0.99902  1.00000       21  Experimental     NaN     NaN  NaN
          352  0.00109  0.00649  0.06306       22  Experimental     NaN     NaN  NaN
          353  0.01528  0.08809  0.60233       23  Experimental     NaN     NaN  NaN
          354  0.00314  0.01865  0.17161       24  Experimental     NaN     NaN  NaN
          355  0.16309  0.65565  0.99998       25  Experimental     NaN     NaN  NaN
          356  0.99842  1.00000  1.00000       26  Experimental     NaN     NaN  NaN
          357  0.00111  0.00663  0.06433       27  Experimental     NaN     NaN  NaN
          358  0.00033  0.00198  0.01961       28  Experiment

Finished evaluating HDX
     Bc   Bh          calc_name
0  0.35  2.0    prior_BPTI_TFES
1  0.35  2.0  train_BPTI_TFES_1
2  0.35  2.0  train_BPTI_TFES_2
3  0.35  2.0  train_BPTI_TFES_3
4  0.35  2.0  train_BPTI_TFES_4
5  0.35  2.0  train_BPTI_TFES_5
Verifiying data
AnalysisInfo(settings_name='BPTI_TFES_RW_bench_R3_k_sequence', analysis_name='RW_bench', n_reps=5, split_mode='R3', times=[0.167, 1, 10], calc_name='BPTI_TFES', expt_name='Experimental', system_name='BPTI_TFES_RW_bench_R3_k_sequence')
train_dfs, 0 verifying...
AnalysisInfo(settings_name='BPTI_TFES_RW_bench_R3_k_sequence', analysis_name='RW_bench', n_reps=5, split_mode='R3', times=[0.167, 1, 10], calc_name='BPTI_TFES', expt_name='Experimental', system_name='BPTI_TFES_RW_bench_R3_k_sequence')
train_dfs, 1 verifying...
AnalysisInfo(settings_name='BPTI_TFES_RW_bench_R3_k_sequence', analysis_name='RW_bench', n_reps=5, split_mode='R3', times=[0.167, 1, 10], calc_name='BPTI_TFES', expt_name='Experimental', system_name='BPTI_TFES_RW_b

In [7]:
print(combined_analysis_dump.keys())

dict_keys(['train_dfs', 'val_dfs', 'expt_df', 'merge_df', 'expt_segs', 'train_segs', 'val_segs', 'HDX_data', 'weights', 'features', 'BV_constants', 'LogPfs', 'analysis_df'])


In [5]:
print(combined_analysis_dump["weights"])

                                              weights          calc_name  \
0   [0.002, 0.002, 0.002, 0.002, 0.002, 0.002, 0.0...    prior_BPTI_TFES   
1   [0.0051564221522796775, 0.0012028067992362933,...  train_BPTI_TFES_1   
2                                                 NaN  train_BPTI_TFES_1   
3   [0.0008733165842455668, 0.0005845037727054736,...  train_BPTI_TFES_2   
4                                                 NaN  train_BPTI_TFES_2   
5   [0.0029737372137037107, 0.0014947655767395829,...  train_BPTI_TFES_3   
6                                                 NaN  train_BPTI_TFES_3   
7   [0.002992764304636476, 0.0011007282884660085, ...  train_BPTI_TFES_4   
8                                                 NaN  train_BPTI_TFES_4   
9   [0.0008559189953014619, 0.0005834270990627614,...  train_BPTI_TFES_5   
10                                                NaN  train_BPTI_TFES_5   

    likelihood                              name  \
0          NaN  BPTI_TFES_RW_bench_

In [6]:
# now save weights to dataframe
